In [1]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import logging  # Biblioteca de logging
from pathlib import Path  # Biblioteca para manipulação de caminhos
from typing import Optional, Dict, List

# --- CONFIGURAÇÃO PRINCIPAL (Ajustes de Análise) ---
# (Suas configurações de METRIC_CONFIG, ALGORITHMS_TO_PROCESS, RENAME_MAP, etc.
# permanecem exatamente como estão aqui)
# 1. MÉTRICAS E SEPARADORES:
METRIC_CONFIG = {
    "results_flows": ",",
    "results_cpu": ",",
    "results_cache": ",",
    "results_bandwidth": ";",
}
# 2. ALGORITMOS:
ALGORITHMS_TO_PROCESS = ["hephaestus", "darsppo", "ga", "kuririnPPO"]
RENAME_MAP = {
    "hephaestus": "Hephaestus",
    "darsppo": "DA-RSPPO",
    "ga": "OSCIM",
    "kuririnPPO": "INOMMUS",
}
CORES_ALGORITMOS = {
    "Hephaestus": "#EF553B",
    "DA-RSPPO": "#00CC96",
    "OSCIM": "#636EFA",
    "INOMMUS": "#AB63FA",
}
# 3. FILTRAGEM:
PLAYER_CONFIG_PATTERN = "_p_6_a_1.0_c_0"
COMPLETION_THRESHOLD_SECONDS = 900
ANALYSIS_START_TIME = 200
# 4. BINS DE TEMPO:
time_bins = [ANALYSIS_START_TIME, 300, 500, 700, 900, float("inf")]
time_labels = [250, 400, 600, 800, 1000]

# --- FIM DA CONFIGURAÇÃO DE ANÁLISE ---

# --- SETUP DE LOGGING E CAMINHOS ---
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
try:
    PROJECT_ROOT = Path(__file__).parent.parent.parent.resolve()
except NameError:
    logging.warning("Executando em modo interativo (ex: Jupyter). Usando Path.cwd() como base.")
    PROJECT_ROOT = Path.cwd().parent.parent.resolve()

BASE_RESULTS_DIR = PROJECT_ROOT / "results"
logging.info(f"Raiz do projeto detectada: {PROJECT_ROOT}")
logging.info(f"Diretório de resultados alvo: {BASE_RESULTS_DIR.resolve()}")

pio.renderers.default = "notebook"
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 1000)

# --- FIM DO SETUP ---


# (A definição da função load_and_filter_metric_data permanece igual)
def load_and_filter_metric_data(
    metric_path: Path,
    metric_name: str,
    algorithms: List[str],
    rename_map: Dict[str, str],
    pattern: str,
    delimiter: str = ",",
    min_time: float = 900,
) -> Optional[pd.DataFrame]:
    # ... (cole sua função load_and_filter_metric_data inteira aqui, sem mudanças)
    logging.info(f"\n--- Carregando dados de: {metric_name} (Separador '{delimiter}') ---")
    all_runs_data = []
    for alg in algorithms:
        pattern1_str = f"{alg}_s_50*{pattern}"
        pattern2_str = f"alg_{alg}_s_50*{pattern}"
        scenario_folders = sorted(
            list(metric_path.glob(pattern1_str)) + list(metric_path.glob(pattern2_str))
        )
        if not scenario_folders:
            logging.warning(f"Nenhuma pasta encontrada para '{alg}' em {metric_path}")
            continue
        for folder in scenario_folders:
            run_files = sorted(folder.glob("*.csv"))
            for run_file in run_files:
                try:
                    df_run = pd.read_csv(run_file, sep=delimiter)
                    if "time_seconds" not in df_run.columns and metric_name == "results_flows":
                        logging.error(f"Arquivo {run_file.name} não tem 'time_seconds'. Pulando.")
                        continue
                    df_run["algoritmo"] = alg
                    df_run["source_file"] = run_file.name
                    all_runs_data.append(df_run)
                except Exception as e:
                    logging.error(f"Erro ao ler {run_file}: {e}")
    if not all_runs_data:
        logging.critical(f"--- ERRO CRÍTICO: Nenhum dado foi carregado de {metric_name} ---")
        return None
    df_metric = pd.concat(all_runs_data, ignore_index=True)
    df_filtered = df_metric.copy()
    if "time_seconds" in df_metric.columns:
        logging.info(
            f"Iniciando filtro de completude (tempo >= {min_time}s) para {metric_name}..."
        )
        max_times_per_file = df_metric.groupby("source_file")["time_seconds"].max()
        incomplete_files = max_times_per_file[max_times_per_file < min_time].index
        if not incomplete_files.empty:
            logging.warning(f"ATENÇÃO: Removendo {len(incomplete_files)} simulações incompletas:")
            for f in incomplete_files:
                logging.warning(f"  -> {f} (max_time: {max_times_per_file[f]:.2f}s)")
            df_filtered = df_metric[~df_metric["source_file"].isin(incomplete_files)].copy()
            logging.info(
                f"Filtro concluído. {len(df_filtered)} de {len(df_metric)} linhas de dados mantidas."
            )
        else:
            logging.info("Nenhuma simulação incompleta encontrada. Todos os dados serão usados.")
    else:
        logging.warning(
            f"'time_seconds' não está em {metric_name}. Não é possível filtrar por completude."
        )
    df_filtered["algoritmo"] = df_filtered["algoritmo"].replace(rename_map)
    logging.info(f"Sucesso! Dados de {metric_name} carregados e filtrados.")
    return df_filtered


# --- FIM DA DEFINIÇÃO DA FUNÇÃO ---


# --- CÓDIGO DE EXECUÇÃO (AGORA NO ESCOPO GLOBAL) ---
# (Este código estava dentro de main() - agora está desidentado)

logging.info(f"Análise configurada para 6 Players (Padrão: '*{PLAYER_CONFIG_PATTERN}')")
logging.info(
    f"ATENÇÃO: Apenas simulações com >= {COMPLETION_THRESHOLD_SECONDS}s serão carregadas."
)
logging.info(f"ATENÇÃO: A análise de tempo irá desconsiderar dados de 0-{ANALYSIS_START_TIME}s.\n")

all_dataframes: Dict[str, Optional[pd.DataFrame]] = {}

if not BASE_RESULTS_DIR.is_dir():
    logging.critical(
        f"ERRO CRÍTICO: O diretório base não foi encontrado: {BASE_RESULTS_DIR.resolve()}"
    )
    logging.critical("Verifique a estrutura de pastas. O script esperava encontrar 'results' em:")
    logging.critical(f"{BASE_RESULTS_DIR.resolve()}")
    # Aqui você pode querer parar a execução, mas em um notebook,
    # os erros de log serão suficientes.
else:
    logging.info(f"Procurando dados em: {BASE_RESULTS_DIR.resolve()}")
    for metric_folder, delimiter in METRIC_CONFIG.items():
        metric_path = BASE_RESULTS_DIR / metric_folder
        if not metric_path.is_dir():
            logging.warning(f"Pasta da métrica não encontrada, pulando: {metric_path}")
            all_dataframes[metric_folder] = None
            continue
        df = load_and_filter_metric_data(
            metric_path=metric_path,
            metric_name=metric_folder,
            algorithms=ALGORITHMS_TO_PROCESS,
            rename_map=RENAME_MAP,
            pattern=PLAYER_CONFIG_PATTERN,
            delimiter=delimiter,
            min_time=COMPLETION_THRESHOLD_SECONDS,
        )
        all_dataframes[metric_folder.replace("results_", "df_")] = df

# --- VERIFICAÇÃO (AGORA NO ESCOPO GLOBAL) ---
logging.info("\n--- Verificação dos DataFrames Carregados (Filtrados) ---")

# CRIA AS VARIÁVEIS GLOBAIS QUE AS OUTRAS CÉLULAS PRECISAM
df_flows = all_dataframes.get("df_flows")
df_cpu = all_dataframes.get("df_cpu")
df_cache = all_dataframes.get("df_cache")
df_bandwidth = all_dataframes.get("df_bandwidth")

if df_flows is not None:
    logging.info(f"df_flows carregado. Algoritmos: {df_flows['algoritmo'].unique()}")
    logging.info(f"Total de runs únicos em df_flows: {len(df_flows['source_file'].unique())}")
else:
    logging.error("ERRO: df_flows não foi carregado.")

if df_cpu is not None:
    logging.info(f"df_cpu carregado. Algoritmos: {df_cpu['algoritmo'].unique()}")
else:
    logging.warning("df_cpu não foi carregado.")

if df_cache is not None:
    logging.info(f"df_cache carregado. Algoritmos: {df_cache['algoritmo'].unique()}")
else:
    logging.warning("df_cache não foi carregado.")

if df_bandwidth is not None:
    logging.info(f"df_bandwidth carregado. Algoritmos: {df_bandwidth['algoritmo'].unique()}")
else:
    logging.warning("df_bandwidth não foi carregado.")

2025-10-29 20:52:00,509 - WARNING - Executando em modo interativo (ex: Jupyter). Usando Path.cwd() como base.
2025-10-29 20:52:00,510 - INFO - Raiz do projeto detectada: /home/jupyter-davidgn/multi-user-sfc
2025-10-29 20:52:00,511 - INFO - Diretório de resultados alvo: /home/jupyter-davidgn/multi-user-sfc/results
2025-10-29 20:52:00,557 - INFO - Análise configurada para 6 Players (Padrão: '*_p_6_a_1.0_c_0')
2025-10-29 20:52:00,557 - INFO - ATENÇÃO: Apenas simulações com >= 900s serão carregadas.
2025-10-29 20:52:00,558 - INFO - ATENÇÃO: A análise de tempo irá desconsiderar dados de 0-200s.

2025-10-29 20:52:00,558 - INFO - Procurando dados em: /home/jupyter-davidgn/multi-user-sfc/results
2025-10-29 20:52:00,559 - INFO - 
--- Carregando dados de: results_flows (Separador ',') ---
2025-10-29 20:52:00,575 - WARNING - Nenhuma pasta encontrada para 'darsppo' em /home/jupyter-davidgn/multi-user-sfc/results/results_flows
2025-10-29 20:52:00,575 - WARNING - Nenhuma pasta encontrada para 'ga' e

In [2]:
import os

# Define o nome da pasta para salvar os PDFs
pdf_output_folder = "article_graphs_pdf"

# Cria a pasta se ela não existir
if not os.path.exists(pdf_output_folder):
    os.makedirs(pdf_output_folder)
    print(f"Pasta '{pdf_output_folder}' criada com sucesso.")
else:
    print(f"Pasta '{pdf_output_folder}' já existe.")

Pasta 'article_graphs_pdf' já existe.


In [3]:
import zipfile
import glob
import os  # Make sure os is imported if running this cell independently

# Nome da pasta onde os PDFs foram salvos
pdf_output_folder = "article_graphs_pdf"

# Nome do arquivo ZIP de saída
zip_filename = "article_graphs.zip"

# Encontra todos os arquivos PDF na pasta de saída
pdf_files_to_zip = glob.glob(os.path.join(pdf_output_folder, "*.pdf"))

if not pdf_files_to_zip:
    print(
        f"Nenhum arquivo PDF encontrado na pasta '{pdf_output_folder}'. O arquivo ZIP não foi criado."
    )  # Translated
else:
    try:
        # Cria o arquivo ZIP em modo de escrita
        with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
            print(f"Criando arquivo ZIP: {zip_filename}")  # Translated
            # Adiciona cada arquivo PDF ao ZIP
            for pdf_file in pdf_files_to_zip:
                # Usa os.path.basename para não incluir o caminho completo dentro do ZIP
                zipf.write(pdf_file, os.path.basename(pdf_file))
                print(f"  -> Adicionando: {os.path.basename(pdf_file)}")  # Translated
        print(
            f"\nArquivo ZIP '{zip_filename}' criado com sucesso contendo {len(pdf_files_to_zip)} gráficos."
        )  # Translated
    except Exception as e:
        print(f"\nERRO ao criar o arquivo ZIP: {e}")  # Translated

Nenhum arquivo PDF encontrado na pasta 'article_graphs_pdf'. O arquivo ZIP não foi criado.


In [4]:
# --- Cell 2 (English): GRAPHS FROM RESULTS_FLOWS [SFC -> MSC + LARGER FONTS] ---

if "df_flows" in locals() and df_flows is not None:
    # --- TRANSLATED METRICS (SFC -> MSC) ---
    metrics_to_plot_flows = {
        "Latency (ms)": "latency",
        "CPU Utilization (%)": "cpu_utilization",
        "Cache Utilization (%)": "cache_utilization",
        "Bandwidth Utilization (%)": "bandwidth_utilization",
        "Acceptance Rate (%)": "acceptance_rate",
        "Decision Time (ms)": "decision_time_ms",
        "Energy Consumption (Watts)": "energy_consumption",
        "Number of Active MSCs": "running_sfcs",  # <<< RENAMED HERE >>>
    }

    algoritmos_presentes_flows = df_flows["algoritmo"].unique()
    print(f"Plotting 'flows' graphs for: {algoritmos_presentes_flows}")

    for y_title_en, column_y in metrics_to_plot_flows.items():
        print(f"Generating 'flows' graph for: {y_title_en}")

        df_plot = df_flows.copy()
        if "time_seconds" in df_plot.columns:
            df_plot = df_plot[df_plot["time_seconds"] > 220].copy()
        if df_plot.empty:
            continue
        if column_y not in df_plot.columns:
            continue
        df_plot[column_y] = pd.to_numeric(df_plot[column_y], errors="coerce")
        subset_cols = [column_y, "time_seconds"]
        # Check uses original column name 'running_sfcs'
        if "running_sfcs" in df_plot.columns and column_y != "running_sfcs":
            subset_cols.append("running_sfcs")
        df_plot.dropna(subset=subset_cols, inplace=True)
        if df_plot.empty:
            continue
        df_plot["time_window"] = pd.cut(
            df_plot["time_seconds"], bins=time_bins, labels=time_labels, right=False
        )
        df_plot.dropna(subset=["time_window"], inplace=True)
        if df_plot.empty:
            continue
        if "(%)" in y_title_en and column_y != "acceptance_rate":
            df_plot[column_y] = df_plot[column_y] * 100

        # Create Figure (logic for decision time vs normal)
        if column_y == "decision_time_ms":
            fig = make_subplots(
                rows=2,
                cols=1,
                shared_xaxes=True,
                vertical_spacing=0.08,
                row_heights=[0.4, 0.6],
                subplot_titles=("Oscim Algorithm (Slow Scale)", "Other Algorithms (Fast Scale)"),
            )
            # ... (add traces for decision time) ...
            df_slow = df_plot[df_plot["algoritmo"] == "Oscim"]
            fig.add_trace(
                go.Box(
                    x=df_slow["time_window"],
                    y=df_slow[column_y],
                    name="Oscim",
                    marker_color=cores_algoritmos.get("Oscim"),
                    boxpoints=False,
                ),
                row=1,
                col=1,
            )
            fast_algs = [alg for alg in algoritmos_presentes_flows if alg != "Oscim"]
            for alg_nome in fast_algs:
                df_alg = df_plot[df_plot["algoritmo"] == alg_nome]
                fig.add_trace(
                    go.Box(
                        x=df_alg["time_window"],
                        y=df_alg[column_y],
                        name=alg_nome,
                        marker_color=cores_algoritmos.get(alg_nome),
                        boxpoints=False,
                    ),
                    row=2,
                    col=1,
                )
            fig.update_layout(height=800, boxmode="group", showlegend=True)
            fig.update_yaxes(title_text=y_title_en, row=1, col=1)
            fig.update_yaxes(title_text=y_title_en, type="log", row=2, col=1)
            fig.update_xaxes(
                title_text="Simulation Time Window (s)",
                categoryorder="array",
                categoryarray=time_labels,
                row=2,
                col=1,
            )
        else:
            fig = make_subplots(specs=[[{"secondary_y": True}]])
            # Uses original column name 'running_sfcs' for grouping and plotting data
            if "running_sfcs" in df_plot.columns and column_y != "running_sfcs":
                df_sfcs_media = (
                    df_plot.groupby("time_window", observed=True)["running_sfcs"]
                    .mean()
                    .reset_index()
                )
                fig.add_trace(
                    go.Scatter(
                        x=df_sfcs_media["time_window"],
                        y=df_sfcs_media["running_sfcs"],
                        # <<< RENAMED LABEL >>>
                        name="Average Active MSCs",
                        mode="lines+markers",
                        line=dict(color="black", width=3),
                    ),
                    secondary_y=True,
                )
                # <<< RENAMED AXIS TITLE >>>
                fig.update_yaxes(title_text="Average Active MSCs", secondary_y=True)
            for alg_nome in algoritmos_presentes_flows:
                df_alg = df_plot[df_plot["algoritmo"] == alg_nome]
                fig.add_trace(
                    go.Box(
                        x=df_alg["time_window"],
                        y=df_alg[column_y],
                        name=alg_nome,
                        marker_color=cores_algoritmos.get(alg_nome),
                        boxpoints=False,
                    ),
                    secondary_y=False,
                )
            fig.update_layout(boxmode="group")
            if "(%)" in y_title_en:
                fig.update_yaxes(title_text=y_title_en, secondary_y=False, ticksuffix="%")
            else:
                fig.update_yaxes(title_text=y_title_en, secondary_y=False)
            fig.update_xaxes(
                title_text="Simulation Time Window (s)",
                categoryorder="array",
                categoryarray=time_labels,
            )

        # --- FINAL STYLING AND EXPORT ---
        main_title = f"Distribution of {y_title_en}"
        # Check uses original column name 'running_sfcs'
        if "running_sfcs" in df_plot.columns and column_y != "running_sfcs":
            main_title += " vs. System Load"

        # PDF filename uses the (potentially renamed) y_title_en
        safe_filename = "".join(c if c.isalnum() else "_" for c in y_title_en)
        pdf_filename = f"graph_flows_{safe_filename}.pdf"
        pdf_filepath = os.path.join(pdf_output_folder, pdf_filename)

        fig.update_layout(
            margin=dict(l=20, r=20, t=60, b=50, pad=0),
            title_text=main_title,
            title_font_size=22,
            title_x=0.5,
            font_size=16,
            legend_title_text="",
            legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
        )

        try:
            fig.write_image(pdf_filepath)
            print(f"   -> Graph saved as PDF: {pdf_filepath}")
        except Exception as e:
            print(f"   -> ERROR saving PDF {pdf_filepath}: {e}")
            print("      Check if 'kaleido' library is installed.")

        fig.show()
else:
    print("DataFrame 'df_flows' not loaded. Skipping Cell 2.")

Plotting 'flows' graphs for: ['Hephaestus']
Generating 'flows' graph for: Latency (ms)


NameError: name 'cores_algoritmos' is not defined

In [ ]:
# --- Cell 3 (English): Graphs "Per Flow" [FINAL VERSION - LARGER FIGURE + SPACING + LEGEND FIX] ---

if "df_flows" in locals() and df_flows is not None:
    pdf_output_folder = "article_graphs_pdf"
    if not os.path.exists(pdf_output_folder):
        os.makedirs(pdf_output_folder)

    print("\nPreparing DataFrame for 'per flow' analysis...")  # Translated

    df_plot_flows_micro = df_flows.copy()
    if "time_seconds" in df_plot_flows_micro.columns:
        df_plot_flows_micro = df_plot_flows_micro[df_plot_flows_micro["time_seconds"] > 220].copy()

    try:
        # Extracts flow type from sfc_id column
        df_plot_flows_micro["flow_type"] = df_plot_flows_micro["sfc_id"].apply(
            lambda x: "_".join(x.split("_")[:2]) if isinstance(x, str) else "unknown"
        )
    except Exception as e:
        print(f"Error creating 'flow_type': {e}")

    if "time_seconds" in df_plot_flows_micro.columns:
        df_plot_flows_micro["time_window"] = pd.cut(
            df_plot_flows_micro["time_seconds"], bins=time_bins, labels=time_labels, right=False
        )
    else:
        print("Error: 'time_seconds' column not found.")

    if "flow_type" in df_plot_flows_micro.columns:
        df_plot_flows_micro.dropna(subset=["time_window"], inplace=True)
        if df_plot_flows_micro.empty:
            print("Warning: No 'per flow' data after 0-220s filter.")
        else:
            flow_types_encontrados = [
                ft for ft in df_plot_flows_micro["flow_type"].unique() if ft != "unknown"
            ]
            print(f"Flow types found: {flow_types_encontrados}")
            if not flow_types_encontrados:
                print("No valid flow types found. Skipping.")
            else:
                algoritmos_presentes_flows = df_plot_flows_micro["algoritmo"].unique()
                metrics_per_flow = {
                    "Latency (ms)": "latency",
                    "Acceptance Rate (%)": "acceptance_rate",
                    "Decision Time (ms)": "decision_time_ms",
                }
                num_rows = len(flow_types_encontrados)

                for y_title_en, column_y in metrics_per_flow.items():
                    print(f"Generating 'per flow' graph for: {y_title_en}")
                    fig = make_subplots(
                        rows=num_rows,
                        cols=1,
                        subplot_titles=flow_types_encontrados,
                        shared_xaxes=True,
                        shared_yaxes=True,
                    )
                    legend_added = set()
                    for i, flow_tipo in enumerate(flow_types_encontrados):
                        row_num = i + 1
                        df_tipo = df_plot_flows_micro[
                            df_plot_flows_micro["flow_type"] == flow_tipo
                        ]
                        for alg_nome in algoritmos_presentes_flows:
                            df_alg = df_tipo[df_tipo["algoritmo"] == alg_nome].copy()
                            if df_alg.empty:
                                continue
                            df_alg[column_y] = pd.to_numeric(df_alg[column_y], errors="coerce")
                            df_alg.dropna(subset=[column_y, "time_window"], inplace=True)
                            if df_alg.empty:
                                continue
                            if "(%)" in y_title_en and column_y != "acceptance_rate":
                                df_alg[column_y] = df_alg[column_y] * 100
                            show_legend_for_alg = alg_nome not in legend_added
                            fig.add_trace(
                                go.Box(
                                    x=df_alg["time_window"],
                                    y=df_alg[column_y],
                                    name=alg_nome,
                                    legendgroup=alg_nome,
                                    showlegend=show_legend_for_alg,
                                    marker_color=cores_algoritmos.get(alg_nome),
                                    boxpoints=False,
                                ),
                                row=row_num,
                                col=1,
                            )
                            if show_legend_for_alg:
                                legend_added.add(alg_nome)

                    main_title = f"Distribution of {y_title_en} (Per Flow Type)"
                    fig.update_layout(
                        height=450 * num_rows,
                        width=1400,
                        title_text=main_title,
                        title_font_size=22,
                        title_x=0.5,
                        font_size=16,
                        boxmode="group",
                        boxgap=0.1,
                        boxgroupgap=0.3,
                        legend_title_text="",
                        xaxis_categoryorder="array",
                        xaxis_categoryarray=time_labels,
                        xaxis2_categoryorder="array",
                        xaxis2_categoryarray=time_labels,
                    )
                    fig.update_xaxes(title_text="Simulation Time Window (s)", row=num_rows, col=1)
                    if "(%)" in y_title_en:
                        fig.update_yaxes(ticksuffix="%")
                    if column_y == "decision_time_ms":
                        print("Applying LOG scale for Decision Time.")
                        fig.update_yaxes(type="log")

                    # --- PDF EXPORT ---
                    safe_filename = "".join(c if c.isalnum() else "_" for c in y_title_en)
                    pdf_filename = f"graph_per_flow_{safe_filename}.pdf"
                    pdf_filepath = os.path.join(pdf_output_folder, pdf_filename)
                    fig.update_layout(margin=dict(l=20, r=20, t=60, b=60, pad=0))
                    try:
                        fig.write_image(pdf_filepath)
                        print(f"   -> Graph saved as PDF: {pdf_filepath}")
                    except Exception as e:
                        print(f"   -> ERROR saving PDF {pdf_filepath}: {e}")
                        print("      Check if 'kaleido' is installed.")

                    fig.show()
    else:
        print("Error: 'flow_type' column not created. Skipping 'per flow' analysis.")
else:
    print("DataFrame 'df_flows' not loaded. Skipping Cell 3.")

In [ ]:
# --- Cell 4 (English): SMOOTHED LINE GRAPH [SFC -> MSC + LARGER FONTS] ---

if "df_flows" in locals() and df_flows is not None and "running_sfcs" in df_flows.columns:
    pdf_output_folder = "article_graphs_pdf"
    if not os.path.exists(pdf_output_folder):
        os.makedirs(pdf_output_folder)

    # <<< RENAMED PRINT >>>
    print("Generating line graph for Average Active MSCs (NORMALIZED and SMOOTHED)...")

    df_plot_line_norm = df_flows.copy()
    if "time_seconds" in df_plot_line_norm.columns:
        df_plot_line_norm = df_plot_line_norm[df_plot_line_norm["time_seconds"] > 220].copy()

    if df_plot_line_norm.empty:
        print("Warning: No data for line graph after 0-220s filter.")
    else:
        global_max_sfcs = df_plot_line_norm["running_sfcs"].max()  # Still use original column max
        if global_max_sfcs == 0:
            print("Peak MSCs is 0. Skipping.")  # Renamed print
        else:
            df_plot_line_norm["normalized_ms"] = (
                df_plot_line_norm["running_sfcs"] / global_max_sfcs
            ) * 100  # New column name just for clarity
            # <<< RENAMED PRINT >>>
            print(f"MSC data normalized against peak value of: {global_max_sfcs} MSCs.")

            df_plot_line_norm["time_bin_10s"] = (df_plot_line_norm["time_seconds"] // 10) * 10
            # Use the new normalized column name
            df_agg = (
                df_plot_line_norm.groupby(["algoritmo", "time_bin_10s"])["normalized_ms"]
                .agg(media="mean")
                .reset_index()
            )

            fig = go.Figure()
            algoritmos_presentes_flows_line = df_agg["algoritmo"].unique()
            SMOOTHING_WINDOW = 5

            for alg_nome in algoritmos_presentes_flows_line:
                df_alg = df_agg[df_agg["algoritmo"] == alg_nome].sort_values("time_bin_10s")
                cor = cores_algoritmos.get(alg_nome)
                df_alg["media_suavizada"] = (
                    df_alg["media"]
                    .rolling(window=SMOOTHING_WINDOW, center=True, min_periods=1)
                    .mean()
                )
                fig.add_trace(
                    go.Scatter(
                        x=df_alg["time_bin_10s"],
                        y=df_alg["media_suavizada"],
                        name=alg_nome,
                        mode="lines",
                        line=dict(color=cor, width=3),
                    )
                )

            # --- LAYOUT WITH LARGER FONTS (SFC -> MSC) ---
            # <<< RENAMED TITLES >>>
            main_title = "Average Active MSCs Over Time (Normalized % of Peak & Smoothed)"
            fig.update_layout(
                title=main_title,
                title_font_size=22,
                title_x=0.5,
                font_size=16,
                xaxis_title="Simulation Time (s)",
                yaxis_title="Avg. Active MSCs (% of Peak)",
                yaxis_ticksuffix="%",
                legend_title_text="",
                hovermode="x unified",
            )
            fig.update_xaxes(range=[200, None])

            # --- PDF EXPORT ---
            # <<< RENAMED FILENAME >>>
            pdf_filename = "graph_line_norm_mscs_smoothed.pdf"
            pdf_filepath = os.path.join(pdf_output_folder, pdf_filename)
            fig.update_layout(
                margin=dict(l=20, r=20, t=60, b=50, pad=0),
                legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
            )
            try:
                fig.write_image(pdf_filepath)
                print(f"   -> Graph saved as PDF: {pdf_filepath}")
            except Exception as e:
                print(f"   -> ERROR saving PDF {pdf_filepath}: {e}")
                print("      Check if 'kaleido' library is installed.")

            fig.show()
else:
    # <<< RENAMED PRINT >>>
    print("Cannot generate MSC line graph. Skipping Cell 4.")

In [ ]:
# --- Cell 6 (English): Cache and Bandwidth Graphs [LARGER FONTS + PDF EXPORT] ---


def process_and_plot_resource(df, resource_name, algorithms_present, complete_files_list=None):
    """
    Auxiliary function - Now includes larger fonts and PDF export.
    """
    pdf_output_folder = "article_graphs_pdf"
    if not os.path.exists(pdf_output_folder):
        os.makedirs(pdf_output_folder)

    print(f"\n--- Processing and plotting for: {resource_name} ---")
    if df is None:
        print("DataFrame not loaded. Skipping.")
        return

    try:
        df_proc_res = df.copy()
        if complete_files_list is not None:
            df_proc_res = df_proc_res[df_proc_res["source_file"].isin(complete_files_list)].copy()
            print(f"Filtering {resource_name} data for complete runs.")
        else:
            print(f"Warning: Cannot filter {resource_name}.")

        control_cols = [
            "timestamp",
            "algoritmo",
            "source_file",
            "timestamp_dt",
            "time_start",
            "time_step_sec",
        ]
        resource_cols = [col for col in df_proc_res.columns if col not in control_cols]
        if not resource_cols:
            print("No resource columns found. Skipping.")
            return
        print(f"Found {len(resource_cols)} resource columns.")

        df_proc_res[f"total_{resource_name}"] = df_proc_res[resource_cols].sum(axis=1)
        df_proc_res["timestamp_dt"] = pd.to_datetime(df_proc_res["timestamp"], unit="s")
        df_proc_res["time_start"] = df_proc_res.groupby("source_file")["timestamp_dt"].transform(
            "min"
        )
        df_proc_res["time_step_sec"] = (
            df_proc_res["timestamp_dt"] - df_proc_res["time_start"]
        ).dt.total_seconds()
        df_proc_res = df_proc_res[df_proc_res["time_step_sec"] > 220].copy()
        df_proc_res["time_window"] = pd.cut(
            df_proc_res["time_step_sec"], bins=time_bins, labels=time_labels, right=False
        )
        df_proc_res.dropna(subset=["time_window"], inplace=True)

        if df_proc_res.empty:
            print(f"Warning: No data for {resource_name} after filters.")
            return

        algorithms_present_after_filter = df_proc_res["algoritmo"].unique()
        print(f"Plotting '{resource_name}' graph for: {algorithms_present_after_filter}")

        fig = go.Figure()
        for alg_nome in algorithms_present_after_filter:
            df_alg = df_proc_res[df_proc_res["algoritmo"] == alg_nome]
            fig.add_trace(
                go.Box(
                    x=df_alg["time_window"],
                    y=df_alg[f"total_{resource_name}"],
                    name=alg_nome,
                    marker_color=cores_algoritmos.get(alg_nome),
                    boxpoints=False,
                )
            )

        main_title = f"Total {resource_name.capitalize()} Utilization vs. System Load"
        fig.update_layout(
            title=main_title,
            title_font_size=22,
            title_x=0.5,
            font_size=16,
            boxmode="group",
            boxgap=0.1,
            boxgroupgap=0.3,
            xaxis_title="Simulation Time Window (s)",
            yaxis_title=f"Total {resource_name.capitalize()} Utilization",
            legend_title_text="",
        )
        fig.update_xaxes(categoryorder="array", categoryarray=time_labels)

        # --- PDF EXPORT ---
        safe_filename = "".join(c if c.isalnum() else "_" for c in resource_name)
        pdf_filename = f"graph_{safe_filename}_vs_time.pdf"
        pdf_filepath = os.path.join(pdf_output_folder, pdf_filename)
        fig.update_layout(
            margin=dict(l=20, r=20, t=60, b=50, pad=0),
            legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
        )
        try:
            fig.write_image(pdf_filepath)
            print(f"   -> Graph saved as PDF: {pdf_filepath}")
        except Exception as e:
            print(f"   -> ERROR saving PDF {pdf_filepath}: {e}")
            print("      Check if 'kaleido' library is installed.")

        fig.show()

    except Exception as e:
        print(f"Error processing {resource_name}: {e}")


complete_files = (
    df_flows["source_file"].unique() if "df_flows" in locals() and df_flows is not None else None
)
if "df_cache" in locals() and df_cache is not None:
    alg_presentes_cache = df_cache["algoritmo"].unique()
    process_and_plot_resource(df_cache, "cache", alg_presentes_cache, complete_files)
else:
    print("DataFrame 'df_cache' not loaded. Skipping Cache graph.")
if "df_bandwidth" in locals() and df_bandwidth is not None:
    alg_presentes_bw = df_bandwidth["algoritmo"].unique()
    process_and_plot_resource(df_bandwidth, "bandwidth", alg_presentes_bw, complete_files)
else:
    print("DataFrame 'df_bandwidth' not loaded. Skipping Bandwidth graph.")

In [ ]:
import zipfile
import glob
import os  # Garante que 'os' está importado

# Nome da pasta onde os PDFs foram salvos
pdf_output_folder = "article_graphs_pdf"

# Nome do arquivo ZIP de saída
zip_filename = "article_graphs.zip"

# Encontra todos os arquivos PDF na pasta de saída
pdf_files_to_zip = glob.glob(os.path.join(pdf_output_folder, "*.pdf"))

if not pdf_files_to_zip:
    print(
        f"Nenhum arquivo PDF encontrado na pasta '{pdf_output_folder}'. O arquivo ZIP não foi criado."
    )
else:
    try:
        # Cria o arquivo ZIP em modo de escrita
        with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
            print(f"Criando arquivo ZIP: {zip_filename}")
            # Adiciona cada arquivo PDF ao ZIP
            for pdf_file in pdf_files_to_zip:
                # Usa os.path.basename para não incluir o caminho completo dentro do ZIP
                zipf.write(pdf_file, os.path.basename(pdf_file))
                print(f"  -> Adicionando: {os.path.basename(pdf_file)}")
        print(
            f"\nArquivo ZIP '{zip_filename}' criado com sucesso contendo {len(pdf_files_to_zip)} gráficos."
        )
    except Exception as e:
        print(f"\nERRO ao criar o arquivo ZIP: {e}")

In [ ]:
# --- Cell 2 (English): GRAPHS FROM RESULTS_FLOWS [FINAL - THRESHOLD IN LEGEND (Correct Method)] ---

if "df_flows" in locals() and df_flows is not None:
    # --- Ensure output folder exists ---
    pdf_output_folder = "article_graphs_pdf"
    if not os.path.exists(pdf_output_folder):
        os.makedirs(pdf_output_folder)

    # --- TRANSLATED METRICS (SFC -> MSC) ---
    metrics_to_plot_flows = {
        "Latency (ms)": "latency",
        "CPU Utilization (%)": "cpu_utilization",
        "Cache Utilization (%)": "cache_utilization",
        "Bandwidth Utilization (%)": "bandwidth_utilization",
        "Acceptance Rate (%)": "acceptance_rate",
        "Decision Time (ms)": "decision_time_ms",
        "Energy Consumption (Watts)": "energy_consumption",
        "Number of Active MSCs": "running_sfcs",
    }

    algoritmos_presentes_flows = df_flows["algoritmo"].unique()
    print(f"Plotting 'flows' graphs for: {algoritmos_presentes_flows}")

    msc_line_color = "steelblue"  # Cor azul para MSC

    for y_title_en, column_y in metrics_to_plot_flows.items():
        print(f"Generating 'flows' graph for: {y_title_en}")

        df_plot = df_flows.copy()
        if "time_seconds" in df_plot.columns:
            df_plot = df_plot[df_plot["time_seconds"] > 220].copy()
        if df_plot.empty:
            continue
        if column_y not in df_plot.columns:
            continue
        df_plot[column_y] = pd.to_numeric(df_plot[column_y], errors="coerce")
        subset_cols = [column_y, "time_seconds"]
        if "running_sfcs" in df_plot.columns and column_y != "running_sfcs":
            subset_cols.append("running_sfcs")
        df_plot.dropna(subset=subset_cols, inplace=True)
        if df_plot.empty:
            continue
        df_plot["time_window"] = pd.cut(
            df_plot["time_seconds"], bins=time_bins, labels=time_labels, right=False
        )
        df_plot.dropna(subset=["time_window"], inplace=True)
        if df_plot.empty:
            continue
        if "(%)" in y_title_en and column_y != "acceptance_rate":
            df_plot[column_y] = df_plot[column_y] * 100

        # Create Figure (logic for decision time vs normal)
        # ... (código que define 'fig' - subplots ou normal - igual ao anterior) ...
        if column_y == "decision_time_ms":
            fig = make_subplots(
                rows=2,
                cols=1,
                shared_xaxes=True,
                vertical_spacing=0.08,
                row_heights=[0.4, 0.6],
                subplot_titles=("OSCIM Algorithm (Slow Scale)", "Other Algorithms (Fast Scale)"),
            )
            # ... (add traces for decision time - igual ao anterior) ...
            df_slow = df_plot[df_plot["algoritmo"] == "OSCIM"]
            fig.add_trace(
                go.Box(
                    x=df_slow["time_window"],
                    y=df_slow[column_y],
                    name="OSCIM",
                    marker_color=cores_algoritmos.get("OSCIM"),
                    boxpoints=False,
                ),
                row=1,
                col=1,
            )
            fast_algs = [alg for alg in algoritmos_presentes_flows if alg != "OSCIM"]
            for alg_nome in fast_algs:
                df_alg = df_plot[df_plot["algoritmo"] == alg_nome]
                fig.add_trace(
                    go.Box(
                        x=df_alg["time_window"],
                        y=df_alg[column_y],
                        name=alg_nome,
                        marker_color=cores_algoritmos.get(alg_nome),
                        boxpoints=False,
                    ),
                    row=2,
                    col=1,
                )
            fig.update_layout(height=800, boxmode="group", showlegend=True)
            fig.update_yaxes(title_text=y_title_en, row=1, col=1)
            fig.update_yaxes(title_text=y_title_en, type="log", row=2, col=1)
            fig.update_xaxes(
                title_text="Simulation Time Window (s)",
                categoryorder="array",
                categoryarray=time_labels,
                row=2,
                col=1,
            )
        else:
            fig = make_subplots(specs=[[{"secondary_y": True}]])
            if "running_sfcs" in df_plot.columns and column_y != "running_sfcs":
                df_sfcs_media = (
                    df_plot.groupby("time_window", observed=True)["running_sfcs"]
                    .mean()
                    .reset_index()
                )
                fig.add_trace(
                    go.Scatter(
                        x=df_sfcs_media["time_window"],
                        y=df_sfcs_media["running_sfcs"],
                        name="Average Active MSCs",
                        mode="lines+markers",
                        line=dict(color=msc_line_color, width=3),
                    ),
                    secondary_y=True,
                )
                fig.update_yaxes(
                    title_text="Average Active MSCs",
                    secondary_y=True,
                    color=msc_line_color,
                    title_font_color=msc_line_color,
                )
            for alg_nome in algoritmos_presentes_flows:
                df_alg = df_plot[df_plot["algoritmo"] == alg_nome]
                fig.add_trace(
                    go.Box(
                        x=df_alg["time_window"],
                        y=df_alg[column_y],
                        name=alg_nome,
                        marker_color=cores_algoritmos.get(alg_nome),
                        boxpoints=False,
                    ),
                    secondary_y=False,
                )
            fig.update_layout(boxmode="group")
            if "(%)" in y_title_en:
                fig.update_yaxes(title_text=y_title_en, secondary_y=False, ticksuffix="%")
            else:
                fig.update_yaxes(title_text=y_title_en, secondary_y=False)
            fig.update_xaxes(
                title_text="Simulation Time Window (s)",
                categoryorder="array",
                categoryarray=time_labels,
            )

        # --- FINAL STYLING AND EXPORT ---
        main_title = f"Distribution of {y_title_en}"
        if column_y == "energy_consumption":
            main_title = "Distribution of Energy Consumption vs. System Load"
        elif "running_sfcs" in df_plot.columns and column_y != "running_sfcs":
            main_title += " vs. System Load"

        # --- AJUSTE THRESHOLD LATÊNCIA ---
        if column_y == "latency":
            # 1. Desenha a linha (sem nome para não tentar ir para legenda)
            fig.add_hline(y=13, line_dash="dash", line_color="firebrick")

            # 2. Adiciona um trace 'fantasma' SÓ para a legenda
            fig.add_trace(
                go.Scatter(
                    x=[None],
                    y=[None],  # Sem dados
                    mode="lines",
                    line=dict(color="firebrick", dash="dash"),  # Mesmo estilo da linha
                    name="Latency Threshold (13 ms)",  # Nome para a legenda
                    showlegend=True,
                )
            )

        safe_filename = "".join(
            c if c.isalnum() else "_" for c in y_title_en.replace("(Watts)", "")
        )
        pdf_filename = f"graph_flows_{safe_filename}.pdf"
        pdf_filepath = os.path.join(pdf_output_folder, pdf_filename)

        # Update layout com fontes grandes e margens
        fig.update_layout(
            margin=dict(l=20, r=20, t=70, b=80, pad=0),  # Margem inferior aumentada para legenda
            title_text=main_title,
            title_font_size=28,
            title_x=0.5,
            font_size=20,
            legend_title_text="",
            legend=dict(
                orientation="h", yanchor="top", y=-0.25, xanchor="center", x=0.5
            ),  # y=-0.25
        )

        try:
            fig.write_image(pdf_filepath)
            print(f"   -> Graph saved as PDF: {pdf_filepath}")
        except Exception as e:
            print(f"   -> ERROR saving PDF {pdf_filepath}: {e}")
            print("      Check if 'kaleido' library is installed.")

        fig.show()
else:
    print("DataFrame 'df_flows' not loaded. Skipping Cell 2.")

In [ ]:
# --- Cell 3 (English): Graphs "Per Flow" [MUCH LARGER FONTS + PDF EXPORT] ---

if "df_flows" in locals() and df_flows is not None:
    pdf_output_folder = "article_graphs_pdf"
    if not os.path.exists(pdf_output_folder):
        os.makedirs(pdf_output_folder)

    print("\nPreparing DataFrame for 'per flow' analysis...")

    df_plot_flows_micro = df_flows.copy()
    if "time_seconds" in df_plot_flows_micro.columns:
        df_plot_flows_micro = df_plot_flows_micro[df_plot_flows_micro["time_seconds"] > 220].copy()

    try:
        df_plot_flows_micro["flow_type"] = df_plot_flows_micro["sfc_id"].apply(
            lambda x: "_".join(x.split("_")[:2]) if isinstance(x, str) else "unknown"
        )
    except Exception as e:
        print(f"Error creating 'flow_type': {e}")

    if "time_seconds" in df_plot_flows_micro.columns:
        df_plot_flows_micro["time_window"] = pd.cut(
            df_plot_flows_micro["time_seconds"], bins=time_bins, labels=time_labels, right=False
        )
    else:
        print("Error: 'time_seconds' column not found.")

    if "flow_type" in df_plot_flows_micro.columns:
        df_plot_flows_micro.dropna(subset=["time_window"], inplace=True)
        if df_plot_flows_micro.empty:
            print("Warning: No 'per flow' data after 0-220s filter.")
        else:
            flow_types_encontrados = [
                ft for ft in df_plot_flows_micro["flow_type"].unique() if ft != "unknown"
            ]
            print(f"Flow types found: {flow_types_encontrados}")
            if not flow_types_encontrados:
                print("No valid flow types found. Skipping.")
            else:
                algoritmos_presentes_flows = df_plot_flows_micro["algoritmo"].unique()
                metrics_per_flow = {
                    "Latency (ms)": "latency",
                    "Acceptance Rate (%)": "acceptance_rate",
                    "Decision Time (ms)": "decision_time_ms",
                }
                num_rows = len(flow_types_encontrados)

                for y_title_en, column_y in metrics_per_flow.items():
                    print(f"Generating 'per flow' graph for: {y_title_en}")
                    fig = make_subplots(
                        rows=num_rows,
                        cols=1,
                        subplot_titles=flow_types_encontrados,
                        shared_xaxes=True,
                        shared_yaxes=True,
                    )
                    legend_added = set()
                    for i, flow_tipo in enumerate(flow_types_encontrados):
                        row_num = i + 1
                        df_tipo = df_plot_flows_micro[
                            df_plot_flows_micro["flow_type"] == flow_tipo
                        ]
                        for alg_nome in algoritmos_presentes_flows:
                            df_alg = df_tipo[df_tipo["algoritmo"] == alg_nome].copy()
                            if df_alg.empty:
                                continue
                            df_alg[column_y] = pd.to_numeric(df_alg[column_y], errors="coerce")
                            df_alg.dropna(subset=[column_y, "time_window"], inplace=True)
                            if df_alg.empty:
                                continue
                            if "(%)" in y_title_en and column_y != "acceptance_rate":
                                df_alg[column_y] = df_alg[column_y] * 100
                            show_legend_for_alg = alg_nome not in legend_added
                            fig.add_trace(
                                go.Box(
                                    x=df_alg["time_window"],
                                    y=df_alg[column_y],
                                    name=alg_nome,
                                    legendgroup=alg_nome,
                                    showlegend=show_legend_for_alg,
                                    marker_color=cores_algoritmos.get(alg_nome),
                                    boxpoints=False,
                                ),
                                row=row_num,
                                col=1,
                            )
                            if show_legend_for_alg:
                                legend_added.add(alg_nome)

                    # <<< AUMENTADO FONTES >>>
                    main_title = f"Distribution of {y_title_en} (Per Flow Type)"
                    fig.update_layout(
                        height=500 * num_rows,  # Aumentado altura por subtrama
                        width=1400,
                        title_text=main_title,
                        title_font_size=28,  # Tamanho do título aumentado
                        title_x=0.5,
                        font_size=20,  # Tamanho base aumentado (eixos, legenda, títulos subplot)
                        boxmode="group",
                        boxgap=0.1,
                        boxgroupgap=0.3,
                        legend_title_text="",
                        xaxis_categoryorder="array",
                        xaxis_categoryarray=time_labels,
                        xaxis2_categoryorder="array",
                        xaxis2_categoryarray=time_labels,
                    )
                    # Ajusta tamanho dos títulos dos subplots (geralmente font_size resolve)
                    fig.update_annotations(font_size=20)  # Força tamanho nos títulos dos subplots

                    fig.update_xaxes(title_text="Simulation Time Window (s)", row=num_rows, col=1)
                    if "(%)" in y_title_en:
                        fig.update_yaxes(ticksuffix="%")
                    if column_y == "decision_time_ms":
                        print("Applying LOG scale for Decision Time.")
                        fig.update_yaxes(type="log")

                    # --- PDF EXPORT ---
                    safe_filename = "".join(c if c.isalnum() else "_" for c in y_title_en)
                    pdf_filename = f"graph_per_flow_{safe_filename}.pdf"
                    pdf_filepath = os.path.join(pdf_output_folder, pdf_filename)
                    fig.update_layout(
                        margin=dict(l=20, r=20, t=70, b=70, pad=0)
                    )  # Margens t/b aumentadas
                    try:
                        fig.write_image(pdf_filepath)
                        print(f"   -> Graph saved as PDF: {pdf_filepath}")
                    except Exception as e:
                        print(f"   -> ERROR saving PDF {pdf_filepath}: {e}")
                        print("      Check if 'kaleido' is installed.")

                    fig.show()
    else:
        print("Error: 'flow_type' column not created. Skipping 'per flow' analysis.")
else:
    print("DataFrame 'df_flows' not loaded. Skipping Cell 3.")

In [ ]:
# --- Cell 4 (English): SMOOTHED LINE GRAPH [MUCH LARGER FONTS + PDF EXPORT] ---

if "df_flows" in locals() and df_flows is not None and "running_sfcs" in df_flows.columns:
    pdf_output_folder = "article_graphs_pdf"
    if not os.path.exists(pdf_output_folder):
        os.makedirs(pdf_output_folder)

    print("Generating line graph for Average Active MSCs (NORMALIZED and SMOOTHED)...")

    df_plot_line_norm = df_flows.copy()
    if "time_seconds" in df_plot_line_norm.columns:
        df_plot_line_norm = df_plot_line_norm[df_plot_line_norm["time_seconds"] > 220].copy()

    if df_plot_line_norm.empty:
        print("Warning: No data for line graph after 0-220s filter.")
    else:
        global_max_sfcs = df_plot_line_norm["running_sfcs"].max()
        if global_max_sfcs == 0:
            print("Peak MSCs is 0. Skipping.")
        else:
            df_plot_line_norm["normalized_ms"] = (
                df_plot_line_norm["running_sfcs"] / global_max_sfcs
            ) * 100
            print(f"MSC data normalized against peak value of: {global_max_sfcs} MSCs.")

            df_plot_line_norm["time_bin_10s"] = (df_plot_line_norm["time_seconds"] // 10) * 10
            df_agg = (
                df_plot_line_norm.groupby(["algoritmo", "time_bin_10s"])["normalized_ms"]
                .agg(media="mean")
                .reset_index()
            )

            fig = go.Figure()
            algoritmos_presentes_flows_line = df_agg["algoritmo"].unique()
            SMOOTHING_WINDOW = 5

            for alg_nome in algoritmos_presentes_flows_line:
                df_alg = df_agg[df_agg["algoritmo"] == alg_nome].sort_values("time_bin_10s")
                cor = cores_algoritmos.get(alg_nome)
                df_alg["media_suavizada"] = (
                    df_alg["media"]
                    .rolling(window=SMOOTHING_WINDOW, center=True, min_periods=1)
                    .mean()
                )
                fig.add_trace(
                    go.Scatter(
                        x=df_alg["time_bin_10s"],
                        y=df_alg["media_suavizada"],
                        name=alg_nome,
                        mode="lines",
                        line=dict(color=cor, width=3),
                    )
                )

            # --- LAYOUT WITH LARGER FONTS ---
            main_title = "Average Active MSCs Over Time (Normalized % of Peak & Smoothed)"
            fig.update_layout(
                title=main_title,
                title_font_size=28,  # <<< AUMENTADO TÍTULO PRINCIPAL >>>
                title_x=0.5,
                font_size=20,  # <<< AUMENTADO TAMANHO BASE (EIXOS, LEGENDA) >>>
                xaxis_title="Simulation Time (s)",
                yaxis_title="Avg. Active MSCs (% of Peak)",
                yaxis_ticksuffix="%",
                legend_title_text="",
                hovermode="x unified",
            )
            fig.update_xaxes(range=[200, None])

            # --- PDF EXPORT ---
            pdf_filename = "graph_line_norm_mscs_smoothed.pdf"
            pdf_filepath = os.path.join(pdf_output_folder, pdf_filename)
            fig.update_layout(
                margin=dict(l=20, r=20, t=70, b=60, pad=0),  # Margens t/b aumentadas
                legend=dict(
                    orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5
                ),  # Legenda mais abaixo
            )
            try:
                fig.write_image(pdf_filepath)
                print(f"   -> Graph saved as PDF: {pdf_filepath}")
            except Exception as e:
                print(f"   -> ERROR saving PDF {pdf_filepath}: {e}")
                print("      Check if 'kaleido' library is installed.")

            fig.show()
else:
    print("Cannot generate MSC line graph. Skipping Cell 4.")

In [ ]:
# --- Cell 5 (English): STACKED BAR CHART [MUCH LARGER FONTS + PDF EXPORT] ---

if "df_flows" in locals() and df_flows is not None:
    pdf_output_folder = "article_graphs_pdf"
    if not os.path.exists(pdf_output_folder):
        os.makedirs(pdf_output_folder)

    print("Processing for Stacked CPU graph (Network vs Mobile)...")

    df_plot_cpu_stacked = df_flows.copy()
    if "time_seconds" in df_plot_cpu_stacked.columns:
        df_plot_cpu_stacked = df_plot_cpu_stacked[df_plot_cpu_stacked["time_seconds"] > 220].copy()

    required_cpu_cols = ["network_cpu_utilization", "mobile_cpu_utilization", "time_seconds"]
    if not all(col in df_plot_cpu_stacked.columns for col in required_cpu_cols):
        print(f"Warning: Skipping Cell 5. Missing columns: {required_cpu_cols}")
    else:
        df_plot_cpu_stacked["network_cpu_utilization"] = pd.to_numeric(
            df_plot_cpu_stacked["network_cpu_utilization"], errors="coerce"
        )
        df_plot_cpu_stacked["mobile_cpu_utilization"] = pd.to_numeric(
            df_plot_cpu_stacked["mobile_cpu_utilization"], errors="coerce"
        )
        df_plot_cpu_stacked["time_window"] = pd.cut(
            df_plot_cpu_stacked["time_seconds"], bins=time_bins, labels=time_labels, right=False
        )
        df_plot_cpu_stacked.dropna(
            subset=["time_window", "network_cpu_utilization", "mobile_cpu_utilization"],
            inplace=True,
        )

        if df_plot_cpu_stacked.empty:
            print("Warning: No CPU data after filters.")
        else:
            df_agg_cpu_stacked = (
                df_plot_cpu_stacked.groupby(["algoritmo", "time_window"], observed=True)[
                    ["network_cpu_utilization", "mobile_cpu_utilization"]
                ]
                .mean()
                .reset_index()
            )

            algoritmos_presentes_cpu_stacked = df_agg_cpu_stacked["algoritmo"].unique()
            print(f"Plotting Stacked CPU graph for: {algoritmos_presentes_cpu_stacked}")

            fig_stacked = go.Figure()
            # ... (código para adicionar traces das barras - igual ao anterior) ...
            for alg_nome in algoritmos_presentes_cpu_stacked:
                df_alg = df_agg_cpu_stacked[df_agg_cpu_stacked["algoritmo"] == alg_nome]
                fig_stacked.add_trace(
                    go.Bar(
                        x=[df_alg["algoritmo"], df_alg["time_window"]],
                        y=df_alg["network_cpu_utilization"],
                        name=f"{alg_nome} - Network",
                        legendgroup=alg_nome,
                        marker_color=cores_algoritmos.get(alg_nome),
                    )
                )
                cor_principal_rgb = tuple(
                    int(cores_algoritmos.get(alg_nome).lstrip("#")[i : i + 2], 16)
                    for i in (0, 2, 4)
                )
                cor_movel = f"rgba({cor_principal_rgb[0]},{cor_principal_rgb[1]},{cor_principal_rgb[2]},0.5)"
                fig_stacked.add_trace(
                    go.Bar(
                        x=[df_alg["algoritmo"], df_alg["time_window"]],
                        y=df_alg["mobile_cpu_utilization"],
                        name=f"{alg_nome} - Mobile",
                        legendgroup=alg_nome,
                        marker_color=cor_movel,
                    )
                )

            # --- LAYOUT WITH LARGER FONTS ---
            main_title = "Average CPU Utilization (Network vs. Mobile) per Algorithm and Time"
            fig_stacked.update_layout(
                barmode="stack",
                title=main_title,
                title_font_size=28,  # <<< AUMENTADO TÍTULO PRINCIPAL >>>
                title_x=0.5,
                font_size=20,  # <<< AUMENTADO TAMANHO BASE (EIXOS, LEGENDA) >>>
                xaxis_title="Algorithm / Simulation Time Window (s)",
                yaxis_title="Average CPU Utilization (%)",
                yaxis_ticksuffix="%",
                legend_title_text="",
                xaxis={
                    "categoryorder": "array",
                    "categoryarray": algoritmos_presentes_cpu_stacked,
                },
            )
            fig_stacked.update_xaxes(tickangle=-45)

            # --- PDF EXPORT ---
            pdf_filename = "graph_cpu_stacked_network_mobile.pdf"
            pdf_filepath = os.path.join(pdf_output_folder, pdf_filename)
            fig_stacked.update_layout(
                margin=dict(
                    l=20, r=20, t=70, b=180, pad=0
                ),  # Margem inferior aumentada AINDA MAIS
                legend=dict(
                    orientation="h", yanchor="top", y=-0.45, xanchor="center", x=0.5
                ),  # Legenda MAIS abaixo
            )
            try:
                fig_stacked.write_image(pdf_filepath)
                print(f"   -> Graph saved as PDF: {pdf_filepath}")
            except Exception as e:
                print(f"   -> ERROR saving PDF {pdf_filepath}: {e}")
                print("      Check if 'kaleido' library is installed.")

            fig_stacked.show()
else:
    print("Cannot generate Stacked CPU graph. Skipping Cell 5.")

In [ ]:
# --- Cell 6 (English): Cache and Bandwidth Graphs [MUCH LARGER FONTS + PDF EXPORT] ---


def process_and_plot_resource(df, resource_name, algorithms_present, complete_files_list=None):
    """
    Auxiliary function - Now includes much larger fonts and PDF export.
    """
    pdf_output_folder = "article_graphs_pdf"
    if not os.path.exists(pdf_output_folder):
        os.makedirs(pdf_output_folder)

    print(f"\n--- Processing and plotting for: {resource_name} ---")
    if df is None:
        print("DataFrame not loaded. Skipping.")
        return

    try:
        df_proc_res = df.copy()
        if complete_files_list is not None:
            df_proc_res = df_proc_res[df_proc_res["source_file"].isin(complete_files_list)].copy()
            print(f"Filtering {resource_name} data for complete runs.")
        else:
            print(f"Warning: Cannot filter {resource_name}.")

        control_cols = [
            "timestamp",
            "algoritmo",
            "source_file",
            "timestamp_dt",
            "time_start",
            "time_step_sec",
        ]
        resource_cols = [col for col in df_proc_res.columns if col not in control_cols]
        if not resource_cols:
            print("No resource columns found. Skipping.")
            return
        print(f"Found {len(resource_cols)} resource columns.")

        df_proc_res[f"total_{resource_name}"] = df_proc_res[resource_cols].sum(axis=1)
        df_proc_res["timestamp_dt"] = pd.to_datetime(df_proc_res["timestamp"], unit="s")
        df_proc_res["time_start"] = df_proc_res.groupby("source_file")["timestamp_dt"].transform(
            "min"
        )
        df_proc_res["time_step_sec"] = (
            df_proc_res["timestamp_dt"] - df_proc_res["time_start"]
        ).dt.total_seconds()
        df_proc_res = df_proc_res[df_proc_res["time_step_sec"] > 220].copy()
        df_proc_res["time_window"] = pd.cut(
            df_proc_res["time_step_sec"], bins=time_bins, labels=time_labels, right=False
        )
        df_proc_res.dropna(subset=["time_window"], inplace=True)

        if df_proc_res.empty:
            print(f"Warning: No data for {resource_name} after filters.")
            return

        algorithms_present_after_filter = df_proc_res["algoritmo"].unique()
        print(f"Plotting '{resource_name}' graph for: {algorithms_present_after_filter}")

        fig = go.Figure()
        for alg_nome in algorithms_present_after_filter:
            df_alg = df_proc_res[df_proc_res["algoritmo"] == alg_nome]
            fig.add_trace(
                go.Box(
                    x=df_alg["time_window"],
                    y=df_alg[f"total_{resource_name}"],
                    name=alg_nome,
                    marker_color=cores_algoritmos.get(alg_nome),
                    boxpoints=False,
                )
            )

        # --- LAYOUT WITH LARGER FONTS ---
        main_title = f"Total {resource_name.capitalize()} Utilization vs. System Load"
        fig.update_layout(
            title=main_title,
            title_font_size=28,  # <<< AUMENTADO TÍTULO PRINCIPAL >>>
            title_x=0.5,
            font_size=20,  # <<< AUMENTADO TAMANHO BASE (EIXOS, LEGENDA) >>>
            boxmode="group",
            boxgap=0.1,
            boxgroupgap=0.3,
            xaxis_title="Simulation Time Window (s)",
            yaxis_title=f"Total {resource_name.capitalize()} Utilization",
            legend_title_text="",
        )
        fig.update_xaxes(categoryorder="array", categoryarray=time_labels)

        # --- PDF EXPORT ---
        safe_filename = "".join(c if c.isalnum() else "_" for c in resource_name)
        pdf_filename = f"graph_{safe_filename}_vs_time.pdf"
        pdf_filepath = os.path.join(pdf_output_folder, pdf_filename)
        fig.update_layout(
            margin=dict(l=20, r=20, t=70, b=60, pad=0),  # Margens t/b aumentadas
            legend=dict(
                orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5
            ),  # Legenda mais abaixo
        )
        try:
            fig.write_image(pdf_filepath)
            print(f"   -> Graph saved as PDF: {pdf_filepath}")
        except Exception as e:
            print(f"   -> ERROR saving PDF {pdf_filepath}: {e}")
            print("      Check if 'kaleido' library is installed.")

        fig.show()

    except Exception as e:
        print(f"Error processing {resource_name}: {e}")


complete_files = (
    df_flows["source_file"].unique() if "df_flows" in locals() and df_flows is not None else None
)
if "df_cache" in locals() and df_cache is not None:
    alg_presentes_cache = df_cache["algoritmo"].unique()
    process_and_plot_resource(df_cache, "cache", alg_presentes_cache, complete_files)
else:
    print("DataFrame 'df_cache' not loaded. Skipping Cache graph.")
if "df_bandwidth" in locals() and df_bandwidth is not None:
    alg_presentes_bw = df_bandwidth["algoritmo"].unique()
    process_and_plot_resource(df_bandwidth, "bandwidth", alg_presentes_bw, complete_files)
else:
    print("DataFrame 'df_bandwidth' not loaded. Skipping Bandwidth graph.")

In [ ]:
# --- Cell 6 (English): Cache and Bandwidth Graphs [MUCH LARGER FONTS + PDF EXPORT] ---


def process_and_plot_resource(df, resource_name, algorithms_present, complete_files_list=None):
    """
    Auxiliary function - Now includes much larger fonts and PDF export.
    """
    pdf_output_folder = "article_graphs_pdf"
    if not os.path.exists(pdf_output_folder):
        os.makedirs(pdf_output_folder)

    print(f"\n--- Processing and plotting for: {resource_name} ---")
    if df is None:
        print("DataFrame not loaded. Skipping.")
        return

    try:
        df_proc_res = df.copy()
        if complete_files_list is not None:
            df_proc_res = df_proc_res[df_proc_res["source_file"].isin(complete_files_list)].copy()
            print(f"Filtering {resource_name} data for complete runs.")
        else:
            print(f"Warning: Cannot filter {resource_name}.")

        control_cols = [
            "timestamp",
            "algoritmo",
            "source_file",
            "timestamp_dt",
            "time_start",
            "time_step_sec",
        ]
        resource_cols = [col for col in df_proc_res.columns if col not in control_cols]
        if not resource_cols:
            print("No resource columns found. Skipping.")
            return
        print(f"Found {len(resource_cols)} resource columns.")

        df_proc_res[f"total_{resource_name}"] = df_proc_res[resource_cols].sum(axis=1)
        df_proc_res["timestamp_dt"] = pd.to_datetime(df_proc_res["timestamp"], unit="s")
        df_proc_res["time_start"] = df_proc_res.groupby("source_file")["timestamp_dt"].transform(
            "min"
        )
        df_proc_res["time_step_sec"] = (
            df_proc_res["timestamp_dt"] - df_proc_res["time_start"]
        ).dt.total_seconds()
        df_proc_res = df_proc_res[df_proc_res["time_step_sec"] > 220].copy()
        df_proc_res["time_window"] = pd.cut(
            df_proc_res["time_step_sec"], bins=time_bins, labels=time_labels, right=False
        )
        df_proc_res.dropna(subset=["time_window"], inplace=True)

        if df_proc_res.empty:
            print(f"Warning: No data for {resource_name} after filters.")
            return

        algorithms_present_after_filter = df_proc_res["algoritmo"].unique()
        print(f"Plotting '{resource_name}' graph for: {algorithms_present_after_filter}")

        fig = go.Figure()
        for alg_nome in algorithms_present_after_filter:
            df_alg = df_proc_res[df_proc_res["algoritmo"] == alg_nome]
            fig.add_trace(
                go.Box(
                    x=df_alg["time_window"],
                    y=df_alg[f"total_{resource_name}"],
                    name=alg_nome,
                    marker_color=cores_algoritmos.get(alg_nome),
                    boxpoints=False,
                )
            )

        # --- LAYOUT WITH LARGER FONTS ---
        main_title = f"Total {resource_name.capitalize()} Utilization vs. System Load"
        fig.update_layout(
            title=main_title,
            title_font_size=28,  # <<< AUMENTADO TÍTULO PRINCIPAL >>>
            title_x=0.5,
            font_size=20,  # <<< AUMENTADO TAMANHO BASE (EIXOS, LEGENDA) >>>
            boxmode="group",
            boxgap=0.1,
            boxgroupgap=0.3,
            xaxis_title="Simulation Time Window (s)",
            yaxis_title=f"Total {resource_name.capitalize()} Utilization",
            legend_title_text="",
        )
        fig.update_xaxes(categoryorder="array", categoryarray=time_labels)

        # --- PDF EXPORT ---
        safe_filename = "".join(c if c.isalnum() else "_" for c in resource_name)
        pdf_filename = f"graph_{safe_filename}_vs_time.pdf"
        pdf_filepath = os.path.join(pdf_output_folder, pdf_filename)
        fig.update_layout(
            margin=dict(l=20, r=20, t=70, b=60, pad=0),  # Margens t/b aumentadas
            legend=dict(
                orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5
            ),  # Legenda mais abaixo
        )
        try:
            fig.write_image(pdf_filepath)
            print(f"   -> Graph saved as PDF: {pdf_filepath}")
        except Exception as e:
            print(f"   -> ERROR saving PDF {pdf_filepath}: {e}")
            print("      Check if 'kaleido' library is installed.")

        fig.show()

    except Exception as e:
        print(f"Error processing {resource_name}: {e}")


complete_files = (
    df_flows["source_file"].unique() if "df_flows" in locals() and df_flows is not None else None
)
if "df_cache" in locals() and df_cache is not None:
    alg_presentes_cache = df_cache["algoritmo"].unique()
    process_and_plot_resource(df_cache, "cache", alg_presentes_cache, complete_files)
else:
    print("DataFrame 'df_cache' not loaded. Skipping Cache graph.")
if "df_bandwidth" in locals() and df_bandwidth is not None:
    alg_presentes_bw = df_bandwidth["algoritmo"].unique()
    process_and_plot_resource(df_bandwidth, "bandwidth", alg_presentes_bw, complete_files)
else:
    print("DataFrame 'df_bandwidth' not loaded. Skipping Bandwidth graph.")

In [ ]:
# --- Cell 7 (English): Graphs of Metric vs. Load [MUCH LARGER FONTS + PDF EXPORT] ---

if "df_flows" in locals() and df_flows is not None:
    pdf_output_folder = "article_graphs_pdf"
    if not os.path.exists(pdf_output_folder):
        os.makedirs(pdf_output_folder)

    print("\n--- Generating Metric vs. MSC Load graphs ---")

    metrics_vs_load = {
        "Latency (ms)": "latency",
        "Acceptance Rate (%)": "acceptance_rate",
        "Energy Consumption (Watts)": "energy_consumption",
        "Decision Time (ms)": "decision_time_ms",
    }

    df_plot_load = df_flows.copy()
    df_plot_load = df_plot_load[df_plot_load["time_seconds"] > 220].copy()

    try:
        load_labels = [
            "0-20% (Very Low)",
            "20-40% (Low)",
            "40-60% (Medium)",
            "60-80% (High)",
            "80-100% (Very High)",
        ]
        df_plot_load["load_window"] = pd.qcut(
            df_plot_load["running_sfcs"], q=5, labels=load_labels, duplicates="drop"
        )
        print("Load windows created successfully.")
        print("Data distribution:")
        print(
            pd.qcut(df_plot_load["running_sfcs"], q=5, duplicates="drop")
            .value_counts()
            .sort_index()
        )
    except Exception as e:
        print(f"Error creating load windows: {e}. Skipping Cell 7.")
        df_plot_load = None

    if df_plot_load is not None:
        algoritmos_presentes_flows = df_plot_load["algoritmo"].unique()

        for y_title_en, column_y in metrics_vs_load.items():
            print(f"Generating graph: {y_title_en} vs. Load")
            fig = go.Figure()

            for alg_nome in algoritmos_presentes_flows:
                df_alg = df_plot_load[df_plot_load["algoritmo"] == alg_nome].copy()
                df_alg[column_y] = pd.to_numeric(df_alg[column_y], errors="coerce")
                df_alg.dropna(subset=[column_y, "load_window"], inplace=True)
                if df_alg.empty:
                    continue
                if "(%)" in y_title_en and column_y != "acceptance_rate":
                    df_alg[column_y] = df_alg[column_y] * 100
                fig.add_trace(
                    go.Box(
                        x=df_alg["load_window"],
                        y=df_alg[column_y],
                        name=alg_nome,
                        marker_color=cores_algoritmos.get(alg_nome),
                        boxpoints=False,
                    )
                )

            # --- LAYOUT WITH LARGER FONTS ---
            main_title = f"Distribution of {y_title_en} vs. System Load (running MSCs)"
            fig.update_layout(
                title=main_title,
                title_font_size=28,  # <<< AUMENTADO TÍTULO PRINCIPAL >>>
                title_x=0.5,
                font_size=20,  # <<< AUMENTADO TAMANHO BASE (EIXOS, LEGENDA) >>>
                boxmode="group",
                boxgap=0.1,
                boxgroupgap=0.3,
                xaxis_title="MSC Load Window (Quintile)",  # Renamed
                yaxis_title=y_title_en,
                legend_title_text="",
                showlegend=True,
                xaxis_categoryorder="array",
                xaxis_categoryarray=load_labels,
            )

            if "(%)" in y_title_en:
                fig.update_yaxes(ticksuffix="%")
            if column_y == "decision_time_ms":
                print("Applying LOG scale for Decision Time.")
                fig.update_yaxes(type="log")

            # --- PDF EXPORT ---
            safe_filename = "".join(c if c.isalnum() else "_" for c in y_title_en)
            pdf_filename = f"graph_vs_msc_load_{safe_filename}.pdf"  # Renamed
            pdf_filepath = os.path.join(pdf_output_folder, pdf_filename)
            fig.update_layout(
                margin=dict(l=20, r=20, t=70, b=60, pad=0),  # Margens t/b aumentadas
                legend=dict(
                    orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5
                ),  # Legenda mais abaixo
            )
            try:
                fig.write_image(pdf_filepath)
                print(f"   -> Graph saved as PDF: {pdf_filepath}")
            except Exception as e:
                print(f"   -> ERROR saving PDF {pdf_filepath}: {e}")
                print("      Check if 'kaleido' library is installed.")

            fig.show()
else:
    print("DataFrame 'df_flows' not loaded. Skipping Cell 7.")

In [ ]:
import zipfile
import glob
import os  # Garante que 'os' está importado

# Nome da pasta onde os PDFs foram salvos
pdf_output_folder = "article_graphs_pdf"

# Nome do arquivo ZIP de saída
zip_filename = "article_graphs.zip"

# Encontra todos os arquivos PDF na pasta de saída
pdf_files_to_zip = glob.glob(os.path.join(pdf_output_folder, "*.pdf"))

if not pdf_files_to_zip:
    print(
        f"Nenhum arquivo PDF encontrado na pasta '{pdf_output_folder}'. O arquivo ZIP não foi criado."
    )
else:
    try:
        # Cria o arquivo ZIP em modo de escrita
        with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
            print(f"Criando arquivo ZIP: {zip_filename}")
            # Adiciona cada arquivo PDF ao ZIP
            for pdf_file in pdf_files_to_zip:
                # Usa os.path.basename para não incluir o caminho completo dentro do ZIP
                zipf.write(pdf_file, os.path.basename(pdf_file))
                print(f"  -> Adicionando: {os.path.basename(pdf_file)}")
        print(
            f"\nArquivo ZIP '{zip_filename}' criado com sucesso contendo {len(pdf_files_to_zip)} gráficos."
        )
    except Exception as e:
        print(f"\nERRO ao criar o arquivo ZIP: {e}")